In [ ]:
!apt-get install openjdk-11-jdk -y
!pip install pyspark

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk-headless openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-

In [49]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
# Configuration
DATA_PATH = "/content/drive/My Drive/ProjectBigData/01 datasets/hospital_prices_clean"
OUTPUT_PATH = "/content/drive/My Drive/ProjectBigData/05 artifacts/model_v3"
MODEL_PATH = f"{OUTPUT_PATH}/hospital_price_pipeline"


In [51]:
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Standard imports
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import warnings
warnings.filterwarnings('ignore')

print("Imports loaded")

Imports loaded


In [52]:
import os
from pyspark.sql import SparkSession
from google.colab import drive
import re
from pyspark.sql import functions as F

In [54]:
spark = (SparkSession.builder
         .appName("HospitalChargesModelV2")
         .config('spark.driver.memory', "8g")
         .config("spark.executor.memory", "8g")
         .getOrCreate())

spark

ConnectionRefusedError: [Errno 111] Connection refused

In [7]:
# Reading the data
hospital_data = spark.read.parquet(DATA_PATH)
print(f"Loaded {hospital_data.count():,} rows")
print("\nSchema:")
hospital_data.printSchema()

print("\nFirst 5 rows:")
hospital_data.show(5, truncate=False)

Loaded 47,224 rows

Schema:
root
 |-- hospital_name: string (nullable = true)
 |-- code_2: string (nullable = true)
 |-- setting: string (nullable = true)
 |-- discounted_cash: double (nullable = true)
 |-- gross_price: double (nullable = true)
 |-- description: string (nullable = true)
 |-- code_1: string (nullable = true)
 |-- code_1_type: string (nullable = true)
 |-- code_2_type: string (nullable = true)
 |-- code_3: string (nullable = true)


First 5 rows:
+-------------+------+----------+---------------+-----------+---------------------------------------------+-----------+-----------+-----------+------+
|hospital_name|code_2|setting   |discounted_cash|gross_price|description                                  |code_1     |code_1_type|code_2_type|code_3|
+-------------+------+----------+---------------+-----------+---------------------------------------------+-----------+-----------+-----------+------+
|massachusetts|0001U |outpatient|987.75         |1317.0     |rbc dna hea 35 ag 11

### Data Preparation for Modelling

In [8]:
# 1. Remove invalid target/gross rows
df_clean = hospital_data.filter(
    (F.col("discounted_cash").isNotNull()) &
    (F.col("discounted_cash") > 0) &
    (F.col("gross_price").isNotNull()) &
    (F.col("gross_price") > 0)
)

print(f"Rows after filtering invalid prices: {df_clean.count():,}")


Rows after filtering invalid prices: 47,224


In [9]:
# 2. Replace missing textual and categorical fields
df_clean = df_clean.fillna({
    "description": "",
    "code_2": "NO_CODE_2",
    "code_2_type": "unknown",
    "code_1": "NONE",
    "code_1_type": "unknown",
    "code_3": "NO_CODE_3",
    "setting": "unknown_setting",
    "hospital_name": "unknown_hospital"
})

# Verify no nulls remain
print("\nNull counts after filling:")
nulls_found = False
for c in df_clean.columns:
    null_count = df_clean.filter(F.col(c).isNull()).count()
    if null_count > 0:
        print(f"  {c}: {null_count}")
        nulls_found = True
if not nulls_found:
    print("No null values found")


Null counts after filling:
No null values found


In [10]:
# 3. Log-transform target and numeric predictor
df_model = (
    df_clean
    .withColumn("log_cash", F.log(F.col("discounted_cash")))
    .withColumn("log_gross", F.log(F.col("gross_price")))
)

print("Log transformation complete")
print(f"\nSample log-transformed values:")
df_model.select("discounted_cash", "log_cash", "gross_price", "log_gross").show(5)

Log transformation complete

Sample log-transformed values:
+---------------+-----------------+-----------+-----------------+
|discounted_cash|         log_cash|gross_price|        log_gross|
+---------------+-----------------+-----------+-----------------+
|         987.75|  6.8954296292915|     1317.0|7.183111701743281|
|          381.0|5.942799375126701|      508.0|6.230481447578482|
|        1301.25|7.171080619929179|     1735.0| 7.45876269238096|
|         6417.0|8.766705997750515|     8556.0|9.054388070202297|
|         6417.0|8.766705997750515|     8556.0|9.054388070202297|
+---------------+-----------------+-----------+-----------------+
only showing top 5 rows



## Train/Validation/Test Split

In [11]:
# Split data: 70% train, 15% validation, 15% test
train_df, val_df, test_df = df_model.randomSplit([0.7, 0.15, 0.15], seed=42)

train_count = train_df.count()
val_count = val_df.count()
test_count = test_df.count()
total_count = train_count + val_count + test_count

print(f"Train set:      {train_count:,} rows ({train_count/total_count*100:.1f}%)")
print(f"Validation set: {val_count:,} rows ({val_count/total_count*100:.1f}%)")
print(f"Test set:       {test_count:,} rows ({test_count/total_count*100:.1f}%)")
print(f"Total:          {total_count:,} rows")


Train set:      33,221 rows (70.3%)
Validation set: 6,980 rows (14.8%)
Test set:       7,023 rows (14.9%)
Total:          47,224 rows


In [12]:
# # Save splits
# splits_path = f"{OUTPUT_PATH}/data_splits"
# os.makedirs(splits_path, exist_ok=True)

# train_df.write.mode("overwrite").parquet(f"{splits_path}/train")
# val_df.write.mode("overwrite").parquet(f"{splits_path}/validation")
# test_df.write.mode("overwrite").parquet(f"{splits_path}/test")

# print(f"Data splits saved to {splits_path}")

## Build Pipeline

In [13]:
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, HashingTF, IDF,
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.regression import GBTRegressor  # Gradient Boosted Trees
# Alternative imports: RandomForestRegressor, DecisionTreeRegressor, LinearRegression
from pyspark.ml import Pipeline

# Define feature columns
text_col = "description"

categorical_cols = [
    "hospital_name",
    "code_2",
    "setting",
    "code_1_type",
    "code_2_type"
]

numeric_cols = ["log_gross"]
target_col = "log_cash"

print("Feature columns defined")
print(f"  Text: {text_col}")
print(f"  Categorical: {categorical_cols}")
print(f"  Numeric: {numeric_cols}")
print(f"  Target: {target_col}")


Feature columns defined
  Text: description
  Categorical: ['hospital_name', 'code_2', 'setting', 'code_1_type', 'code_2_type']
  Numeric: ['log_gross']
  Target: log_cash


In [14]:
# Text preprocessing: TF-IDF
tokenizer = Tokenizer(inputCol=text_col, outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashing_tf = HashingTF(inputCol="filtered", outputCol="tf_raw", numFeatures=20000)
idf = IDF(inputCol="tf_raw", outputCol="tfidf_features")

print("Text preprocessing stages defined")

Text preprocessing stages defined


In [15]:
# Categorical variables: StringIndexer + OneHotEncoder
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_vec"])
    for c in categorical_cols
]

print(f"Categorical encoding stages defined ({len(indexers)} indexers, {len(encoders)} encoders)")

Categorical encoding stages defined (5 indexers, 5 encoders)


In [16]:
# Numeric variables: StandardScaler
num_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="num_unscaled"
)

scaler = StandardScaler(
    inputCol="num_unscaled",
    outputCol="num_scaled",
    withMean=True,
    withStd=True
)

print("Numeric scaling stages defined")

Numeric scaling stages defined


In [17]:
# Assembling final feature vector
feature_cols = ["tfidf_features"] + [f"{c}_vec" for c in categorical_cols] + ["num_scaled"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

print(f"Feature assembler defined")
print(f"Feature columns: {len(feature_cols)} components")

Feature assembler defined
Feature columns: 7 components


In [ ]:
# # Define Linear Regression model
# lr = LinearRegression(
#     featuresCol="features",
#     labelCol=target_col,
#     maxIter=50,
#     regParam=0.1,
#     elasticNetParam=0.0,
#     solver="l-bfgs"  # Better for large datasets
# )

# print("Linear Regression model defined")

In [18]:
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=target_col,
    maxDepth=5,           # Maximum depth of trees
    maxIter=50,           # Number of trees in the ensemble
    stepSize=0.1,         # Learning rate (shrinkage)
    subsamplingRate=0.8,  # Fraction of training data used per tree
    minInstancesPerNode=10,  # Minimum instances per leaf
    maxBins=32,           # Number of bins for discretizing continuous features
    seed=42               # Random seed for reproducibility
)

print("Gradient Boosted Trees (GBT) Regressor model defined")
print("  - Ensemble of 50 trees")
print("  - Max depth: 5")
print("  - Learning rate: 0.1")


Gradient Boosted Trees (GBT) Regressor model defined
  - Ensemble of 50 trees
  - Max depth: 5
  - Learning rate: 0.1


In [19]:
# Build complete pipeline
pipeline = Pipeline(stages=
    [tokenizer, remover, hashing_tf, idf] +
    indexers + encoders +
    [num_assembler, scaler, assembler, gbt]
)

print("Complete ML pipeline built")
print(f"  Total stages: {len(pipeline.getStages())}")
print(f"  Model: Gradient Boosted Trees (GBT) Regressor")

Complete ML pipeline built
  Total stages: 18
  Model: Gradient Boosted Trees (GBT) Regressor


## Train Model

In [ ]:
## Train Model

In [ ]:
print("-" * 80)
print("TRAINING MODEL")
print("-" * 80)
print("\nFitting preprocessing pipeline...")

# Fit pipeline on training data
pipeline_model = pipeline.fit(train_df)

print("Pipeline fitted successfully")

--------------------------------------------------------------------------------
TRAINING MODEL
--------------------------------------------------------------------------------

Fitting preprocessing pipeline...
Pipeline fitted successfully


In [ ]:
print("\nGenerating predictions...")

# Transform all splits
train_pred = pipeline_model.transform(train_df)
val_pred = pipeline_model.transform(val_df)
test_pred = pipeline_model.transform(test_df)

print("Predictions generated for all splits")


Generating predictions...
Predictions generated for all splits


## Evaluate Model

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate(df, label="log_cash"):
    """Evaluate model performance"""
    evaluator = RegressionEvaluator(labelCol=label, predictionCol="prediction")

    rmse = evaluator.setMetricName("rmse").evaluate(df)
    mae = evaluator.setMetricName("mae").evaluate(df)
    r2 = evaluator.setMetricName("r2").evaluate(df)

    # Calculate MAPE manually (on original scale, not log scale)
    df_m = df.withColumn("ape", F.abs((F.exp(F.col("prediction")) - F.exp(F.col(label))) / F.exp(F.col(label))))
    mape = df_m.agg(F.mean("ape")).first()[0]

    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

print("Evaluating model performance...")
print("\n" + "=" * 80)

train_metrics = evaluate(train_pred)
print("TRAIN Metrics:")
for metric, value in train_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

print("\n" + "-" * 80)

val_metrics = evaluate(val_pred)
print("VALIDATION Metrics:")
for metric, value in val_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

print("\n" + "-" * 80)

test_metrics = evaluate(test_pred)
print("TEST Metrics:")
for metric, value in test_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")


Evaluating model performance...

TRAIN Metrics:
  RMSE: 0.2406
  MAE: 0.1140
  R2: 0.9868
  MAPE: 17.66%

--------------------------------------------------------------------------------
VALIDATION Metrics:
  RMSE: 0.2786
  MAE: 0.1195
  R2: 0.9822
  MAPE: 30.84%

--------------------------------------------------------------------------------
TEST Metrics:
  RMSE: 0.2309
  MAE: 0.1151
  R2: 0.9877
  MAPE: 19.17%


## Save Model

In [ ]:
# Save the trained pipeline model
#print(f"Saving model to {MODEL_PATH}...")
#pipeline_model.write().overwrite().save(MODEL_PATH)
#print(f"Model saved successfully to {MODEL_PATH}")

Saving model to /content/drive/My Drive/ProjectBigData/05 artifacts/model_v2/hospital_price_pipeline...
Model saved successfully to /content/drive/My Drive/ProjectBigData/05 artifacts/model_v2/hospital_price_pipeline


In [ ]:
## Linear Regression results
# print("TRAIN:", evaluate(train_pred))
# print("VALID:", evaluate(val_pred))
# print("TEST:", evaluate(test_pred))


TRAIN: {'RMSE': 0.17508234667697603, 'MAE': 0.0823278982612729, 'R2': 0.9930655054521644, 'MAPE': 0.0968552143378268}
VALID: {'RMSE': 0.2811988415004916, 'MAE': 0.12550898219485113, 'R2': 0.9813201149550008, 'MAPE': 0.24203867231511486}
TEST: {'RMSE': 0.2732444927036042, 'MAE': 0.12514627550987256, 'R2': 0.982796110234425, 'MAPE': 0.23323983126728431}


### Build and Evaluate XGBoost Model

In [33]:
# Define new paths for the XGBoost model
XGBOOST_OUTPUT_PATH = '/content/drive/My Drive/ProjectBigData/05 artifacts/model_v3_xgboost'
XGBOOST_MODEL_PATH = f"{XGBOOST_OUTPUT_PATH}/hospital_price_pipeline_xgoost"
os.makedirs(XGBOOST_OUTPUT_PATH, exist_ok=True)
print(f"XGBoost model output path: {XGBOOST_OUTPUT_PATH}")

XGBoost model output path: /content/drive/My Drive/ProjectBigData/05 artifacts/model_v3_xgboost


In [34]:
# Install xgboost
!pip install xgboost

In [35]:
# Import SparkXGBRegressor
from xgboost.spark import SparkXGBRegressor

# Define XGBoost Regressor model
# Using similar parameters to the GBTRegressor for comparison
xgb_regressor = SparkXGBRegressor(
    features_col="features",
    label_col=target_col,
    n_estimators=50,       # Number of trees
    max_depth=5,           # Maximum depth of trees
    learning_rate=0.1,     # Step size shrinkage
    seed=42
)

print("XGBoost Regressor model defined")
print("  - Ensemble of 50 trees")
print("  - Max depth: 5")
print("  - Learning rate: 0.1")

XGBoost Regressor model defined
  - Ensemble of 50 trees
  - Max depth: 5
  - Learning rate: 0.1


In [36]:
# Build complete pipeline with XGBoost
xgb_pipeline = Pipeline(stages=
    [tokenizer, remover, hashing_tf, idf] +
    indexers + encoders +
    [num_assembler, scaler, assembler, xgb_regressor]
)

print("Complete ML pipeline with XGBoost built")
print(f"  Total stages: {len(xgb_pipeline.getStages())}")
print(f"  Model: XGBoost Regressor")

Complete ML pipeline with XGBoost built
  Total stages: 18
  Model: XGBoost Regressor


In [37]:
print("-" * 80)
print("TRAINING XGBOOST MODEL")
print("-" * 80)
print("\nFitting preprocessing pipeline with XGBoost...")

# Fit pipeline on training data
xgb_pipeline_model = xgb_pipeline.fit(train_df)

print("XGBoost Pipeline fitted successfully")

--------------------------------------------------------------------------------
TRAINING XGBOOST MODEL
--------------------------------------------------------------------------------

Fitting preprocessing pipeline with XGBoost...


ConnectionRefusedError: [Errno 111] Connection refused

In [38]:
print("\nGenerating predictions with XGBoost model...")

# Transform all splits with the XGBoost model
xgb_train_pred = xgb_pipeline_model.transform(train_df)
xgb_val_pred = xgb_pipeline_model.transform(val_df)
xgb_test_pred = xgb_pipeline_model.transform(test_df)

print("Predictions generated for all splits using XGBoost model")


Generating predictions with XGBoost model...


ConnectionRefusedError: [Errno 111] Connection refused

In [39]:
spark.stop()

spark = (SparkSession.builder
         .appName("HospitalChargesModelV2")
         .config('spark.driver.memory', "8g")
         .config("spark.executor.memory", "8g")
         .getOrCreate())

ConnectionRefusedError: [Errno 111] Connection refused

In [27]:
xgb_train_pred.printSchema()
xgb_train_pred.show(5)


root
 |-- hospital_name: string (nullable = false)
 |-- code_2: string (nullable = false)
 |-- setting: string (nullable = false)
 |-- discounted_cash: double (nullable = true)
 |-- gross_price: double (nullable = true)
 |-- description: string (nullable = false)
 |-- code_1: string (nullable = false)
 |-- code_1_type: string (nullable = false)
 |-- code_2_type: string (nullable = false)
 |-- code_3: string (nullable = false)
 |-- log_cash: double (nullable = true)
 |-- log_gross: double (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tf_raw: vector (nullable = true)
 |-- tfidf_features: vector (nullable = true)
 |-- hospital_name_idx: double (nullable = false)
 |-- code_2_idx: double (nullable = false)
 |-- setting_idx: double (nullable = false)
 |-- code_1_type_idx: double (nullable = false)
 |-- code_2_type_idx: double (nullable = false)
 

In [32]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F

def evaluate(df, label="log_cash"):
    """Evaluate model performance"""
    evaluator = RegressionEvaluator(labelCol=label, predictionCol="prediction")

    rmse = evaluator.setMetricName("rmse").evaluate(df)
    mae = evaluator.setMetricName("mae").evaluate(df)
    r2 = evaluator.setMetricName("r2").evaluate(df)

    # Calculate MAPE manually (on original scale, not log scale)
    df_m = df.withColumn("ape", F.abs((F.exp(F.col("prediction")) - F.exp(F.col(label))) / F.exp(F.col(label))))
    mape = df_m.agg(F.mean("ape")).first()[0]

    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

print("Evaluating XGBoost model performance...")
print("\n" + "=" * 80)

xgb_train_metrics = evaluate(xgb_train_pred)
print("XGBoost TRAIN Metrics:")
for metric, value in xgb_train_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

print("\n" + "-" * 80)

xgb_val_metrics = evaluate(xgb_val_pred)
print("XGBoost VALIDATION Metrics:")
for metric, value in xgb_val_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

print("\n" + "-" * 80)

xgb_test_metrics = evaluate(xgb_test_pred)
print("XGBoost TEST Metrics:")
for metric, value in xgb_test_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

Evaluating XGBoost model performance...



ConnectionRefusedError: [Errno 111] Connection refused

In [29]:
print("Evaluating XGBoost model performance...")
print("\n" + "=" * 80)

xgb_train_metrics = evaluate(xgb_train_pred)
print("XGBoost TRAIN Metrics:")
for metric, value in xgb_train_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

print("\n" + "-" * 80)

xgb_val_metrics = evaluate(xgb_val_pred)
print("XGBoost VALIDATION Metrics:")
for metric, value in xgb_val_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

print("\n" + "-" * 80)

xgb_test_metrics = evaluate(xgb_test_pred)
print("XGBoost TEST Metrics:")
for metric, value in xgb_test_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

Evaluating XGBoost model performance...



NameError: name 'evaluate' is not defined

In [41]:
try:
    spark.range(1).collect()
    print("Spark session is active and responsive!")
except Exception as e:
    print(f"Spark session is NOT fully functional. Error: {e}")
    print("Please ensure you have restarted the Colab runtime and re-run all cells from the beginning.")

Spark session is NOT fully functional. Error: 'NoneType' object has no attribute 'sc'
Please ensure you have restarted the Colab runtime and re-run all cells from the beginning.


In [40]:
spark

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
# Save the trained XGBoost pipeline model
print(f"Saving XGBoost model to {XGBOOST_MODEL_PATH}...")
xgb_pipeline_model.write().overwrite().save(XGBOOST_MODEL_PATH)
print(f"XGBoost Model saved successfully to {XGBOOST_MODEL_PATH}")

Saving XGBoost model to /content/drive/My Drive/ProjectBigData/05 artifacts/model_v3_xgboost/hospital_price_pipeline_xgboost...


ConnectionRefusedError: [Errno 111] Connection refused